# Spindle-token Databricks Bootstrap Guide for Spark 4

This notebook is intended for Databricks clusters running Spark 4.x.
It installs `spindle-token[spark]` directly from GitHub so you can test changes
before publishing a release.

If you are testing an unpublished branch, change the install cell to a
branch name or commit SHA.

## Prerequisites

This notebook expects a Databricks secret scope named `spindle-token`
with secrets named `PrivateKey` and `MyPartnerPublicKey`.

## String Date Inputs

This Spark 4 notebook intentionally uses string `birth_date` values.
The library should normalize those strings safely, even when Spark 4 is
running with stricter date parsing behavior.

In [ ]:
%pip install "spindle-token[spark] @ git+https://github.com/spindle-health/spindle-token.git@main"

In [ ]:
import sys

import numpy as np
import pandas as pd
import pyarrow as pa
import pyspark

from spindle_token import __version__ as spindle_token_version

expected_spark_prefix = "4."
assert spark.version.startswith(expected_spark_prefix), (
    f"Expected Spark 4.x, found {spark.version}"
)
assert pyspark.__version__.startswith(expected_spark_prefix), (
    f"Expected PySpark 4.x, found {pyspark.__version__}"
)

print(f"Spark version: {spark.version}")
print(f"PySpark version: {pyspark.__version__}")
print(f"spindle-token version: {spindle_token_version}")
print(f"Python version: {sys.version}")
print(f"pandas version: {pd.__version__}")
print(f"pyarrow version: {pa.__version__}")
print(f"numpy version: {np.__version__}")

In [ ]:
from pyspark.sql.types import StructField, StructType, StringType, IntegerType

pii = spark.createDataFrame(
    [
        (1, "Jonas", "Salk", "male", "1914-10-28"),
        (1, "jonas", "salk", "M", "1914-10-28"),
        (2, "Elizabeth", "Blackwell", "F", "1821-02-03"),
        (3, "Edward", "Jenner", "m", "1749-05-17"),
        (4, "Alexander", "Fleming", "M", "1881-08-06"),
    ],
    StructType(
        [
            StructField("label", IntegerType(), True),
            StructField("first_name", StringType(), True),
            StructField("last_name", StringType(), True),
            StructField("gender", StringType(), True),
            StructField("birth_date", StringType(), True),
        ]
    ),
)

pii.printSchema()
display(pii)

In [ ]:
from spindle_token import tokenize
from spindle_token.opprl import OpprlV2

tokens = tokenize(
    pii,
    col_mapping={
        OpprlV2.first_name: "first_name",
        OpprlV2.last_name: "last_name",
        OpprlV2.gender: "gender",
        OpprlV2.birth_date: "birth_date",
    },
    tokens=[OpprlV2.token1, OpprlV2.token2, OpprlV2.token3],
    private_key=dbutils.secrets.getBytes("spindle-token", "PrivateKey"),
)
display(tokens)

In [ ]:
from spindle_token import transcode_out
from spindle_token.opprl import OpprlV2

ephemeral_tokens = transcode_out(
    tokens,
    tokens=(OpprlV2.token1, OpprlV2.token2, OpprlV2.token3),
    recipient_public_key=dbutils.secrets.getBytes("spindle-token", "MyPartnerPublicKey"),
    private_key=dbutils.secrets.getBytes("spindle-token", "PrivateKey"),
).drop("first_name", "last_name", "birth_date")
display(ephemeral_tokens)

In [ ]:
from spindle_token import transcode_in
from spindle_token.opprl import OpprlV2

tokens2 = transcode_in(
    ephemeral_tokens,
    tokens=(OpprlV2.token1, OpprlV2.token2, OpprlV2.token3),
    private_key=dbutils.secrets.getBytes("spindle-token", "PrivateKey"),
)
display(tokens2)